# BBC News Sport Subcategory Classifier

This notebook implements the full subcategory classification pipeline for the **Sport** category of the BBC News dataset.

**Pipeline Overview:**
- Phase 1 — Data Ingestion
- Phase 2 — Text Cleaning
- Phase 3 — Vectorization (Jina AI)
- Phase 4 — UMAP + HDBSCAN Clustering
- Phase 5 — c-TF-IDF + KeyBERT Labeling
- Phase 5B — Verification Block
- Phase 6 — AI Subcategory Naming
- Phase 8 — Manual Noise Correction
- Phase 9 — Named Entity Recognition (NER) with April Events Extraction

## Project Overview
This notebook classifies 511 BBC Sport articles into meaningful subcategories, Beyond subcategory classification, the pipeline also performs Named Entity Recognition (NER) to extract media personalities and their roles, and identifies any events that took place or were scheduled to take place in April. using Jina Embeddings v5, UMAP dimensionality reduction, and HDBSCAN density-based clustering.

**Author:** BBC NLP Project 2  
**Dataset:** 511 BBC Sport Articles  
**Model:** Jina Embeddings v5 (text-small) with clustering adapter

---

## Phase 1: Data Ingestion & Structural Partitioning
In this phase we load all 511 sport articles from the `./sport` folder into a structured DataFrame. Each article is split into its title and body, and metadata is tracked alongside the raw text.

In [1]:
# ============================================================
# PHASE 1: DATA INGESTION & STRUCTURAL PARTITIONING
# ============================================================

import os
import pandas as pd

# 1. Define the path to the sport folder
folder_path = './sport'

# 2. Initialize our data collection list
all_articles = []

# 3. Loop through each .txt file in the sport folder
for filename in sorted(os.listdir(folder_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().strip()

        # 4. Split the file into title (first line) and body (rest)
        lines = content.split('\n')
        title = lines[0].strip()
        body = '\n'.join(lines[1:]).strip()

        # 5. Track metadata alongside raw text
        all_articles.append({
            'file_name': filename,
            'article_id': int(filename.replace('.txt', '')),
            'title': title,
            'raw_text': body,
            'article_length': len(body.split())
        })

# 6. Load into a structured DataFrame
df = pd.DataFrame(all_articles)

# 7. Quick sanity check
print(f" Total articles loaded: {len(df)}")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample:")
df.head()

 Total articles loaded: 511
 Columns: ['file_name', 'article_id', 'title', 'raw_text', 'article_length']

 Sample:


,file_name,article_id,title,raw_text,article_length
0,001.txt,1,Claxton hunting first major medal,British hurdler Sarah Claxton is confident she...,206
1,002.txt,2,O'Sullivan could run in Worlds,Sonia O'Sullivan has indicated that she would ...,139
2,003.txt,3,Greene sets sights on world title,Maurice Greene aims to wipe out the pain of lo...,371
3,004.txt,4,IAAF launches fight against drugs,The IAAF - athletics' world governing body - h...,190
4,005.txt,5,"Dibaba breaks 5,000m world record",Ethiopia's Tirunesh Dibaba set a new world rec...,156


## Phase 2: The Refinement Layer (Data Cleaning)

In this phase we clean all sport articles to prepare them for vectorization.
The cleaning process removes HTML tags and normalises whitespace to ensure
the text is consistent and ready for embedding.

In [15]:
# ============================================================
# PHASE 2: DATA CLEANING — SPORT
# ============================================================
import re
import pandas as pd

def clean_article(text):

    # 1. HTML REMOVAL - confirmed present in raw BBC data
    text = re.sub(r'<[^>]+>', '', text)

    # 2. WHITESPACE CLEANUP
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

df = pd.read_csv('sport_subcategories_output.csv')

if 'raw_text' not in df.columns:
    df = df.rename(columns={'cleaned_text': 'raw_text'})

df['cleaned_text'] = df['raw_text'].apply(clean_article)
df = df.drop(columns=['raw_text'])

df.to_csv('sport_subcategories_output.csv', index=False)

print(f"Total articles: {len(df)}")
print(f"\nSport cleaning complete!")

Total articles: 511

Sport cleaning complete!


## Phase 3: Vectorization Layer (Jina-v5 Intelligence)
In this phase we convert each cleaned article into a 1024-dimensional embedding vector using Jina AI's latest embedding model with the clustering adapter. Articles are processed in batches of 20 with auto-retry on rate limit errors.

In [3]:
# ============================================================
# PHASE 3: THE VECTORIZATION LAYER (JINA-V5) - RATE LIMIT FIX
# ============================================================

import requests
import numpy as np
import time
import os

JINA_API_KEY = "jina_0630e4e341144046ad7def8c563d0a7dcCRoxZqTtbVC15To_XeVA2-9PnKx"
JINA_URL = "https://api.jina.ai/v1/embeddings"

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

def get_embeddings(texts, batch_size=20):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_num = (i // batch_size) + 1

        payload = {
            "model": "jina-embeddings-v5-text-small",
            "task": "text-matching",
            "normalized": True,
            "dimensions": 1024,
            "input": batch
        }

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                response = requests.post(JINA_URL, headers=HEADERS, json=payload)
                response.raise_for_status()
                data = response.json()

                batch_embeddings = [item['embedding'] for item in data['data']]
                all_embeddings.extend(batch_embeddings)

                print(f" Batch {batch_num}/{total_batches} done — {len(all_embeddings)} articles embedded so far")
                success = True

                time.sleep(3)

            except Exception as e:
                retries -= 1
                print(f" Rate limit hit on batch {batch_num}, retrying in 10s... ({retries} retries left)")
                time.sleep(10)

        if not success:
            print(f" Failed on batch {batch_num} after all retries")
            break

    return np.array(all_embeddings)

# Only embed if embeddings don't already exist
if os.path.exists('sport_embeddings.npy'):
    print(" Embeddings already exist — skipping API call")
    embeddings = np.load('sport_embeddings.npy')
else:
    # Run on ALL 511 articles using title + content combined
    texts = (df['title'] + " " + df['cleaned_text']).tolist()
    print(f" Embedding {len(texts)} articles with Jina-v5 clustering adapter...\n")

    embeddings = get_embeddings(texts)

    # Save embeddings to disk permanently
    np.save('sport_embeddings.npy', embeddings)
    print(f"\n Embedding complete!")

print(f" Embedding matrix shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} articles x {embeddings.shape[1]} dimensions")

 Embeddings already exist — skipping API call
 Embedding matrix shape: (511, 1024)
   → 511 articles x 1024 dimensions


## Phase 4: Dimensionality Reduction & Clustering
In this phase we reduce the 1024-dimensional embeddings to 3 dimensions using UMAP while preserving thematic relationships between articles. HDBSCAN then discovers the natural number of subcategories automatically without any pre-set cluster count. Results are saved to disk permanently to ensure reproducibility.

In [4]:
# ============================================================
# PHASE 4: DIMENSIONALITY REDUCTION & CLUSTERING
# ============================================================

import umap
import hdbscan
import numpy as np
import os

# Load embeddings from disk if available
if os.path.exists('sport_embeddings.npy'):
    print(" Loading saved embeddings...")
    embeddings = np.load('sport_embeddings.npy')
    print(f" Embeddings shape: {embeddings.shape}")

# Check if we already have saved cluster results
if os.path.exists('sport_cluster_labels.npy') and os.path.exists('sport_reduced_embeddings.npy'):
    
    print(" Loading saved cluster results...")
    cluster_labels = np.load('sport_cluster_labels.npy')
    reduced_embeddings = np.load('sport_reduced_embeddings.npy')
    confidence_scores = np.load('sport_confidence_scores.npy')
    
    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

else:
    
    # 1. UMAP
    print(" Running UMAP dimensionality reduction...")
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )
    reduced_embeddings = reducer.fit_transform(embeddings)
    print(f" UMAP complete! Shape: {reduced_embeddings.shape}\n")

    # 2. HDBSCAN
    print(" Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=8,
        min_samples=4,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    confidence_scores = clusterer.probabilities_
    confidence_scores[cluster_labels == -1] = 0.0

    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    # Save to disk permanently
    np.save('sport_cluster_labels.npy', cluster_labels)
    np.save('sport_reduced_embeddings.npy', reduced_embeddings)
    np.save('sport_confidence_scores.npy', confidence_scores)
    print(" Results saved to disk permanently!")

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" HDBSCAN complete!")
    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

# Cluster distribution
print(f"\n Articles per subcategory:")
print(df['subcategory_id'].value_counts().sort_index())

 Loading saved embeddings...
 Embeddings shape: (511, 1024)
 Loading saved cluster results...
 Subcategories found: 14
  Noise articles: 52
 Clustered articles: 459

 Articles per subcategory:
subcategory_id
-1     52
 0     66
 1     88
 2     10
 3     12
 4     29
 5     17
 6     10
 7     52
 8     68
 9     22
 10    33
 11    19
 12     9
 13    24
Name: count, dtype: int64


## Phase 5: Thematic Labeling (c-TF-IDF)
In this phase we extract the most unique and discriminative keywords for each cluster using c-TF-IDF. This compares word frequencies across clusters to surface words that are characteristic of each subcategory rather than just globally common words.

In [5]:
# ============================================================
# PHASE 5: THEMATIC LABELING (c-TF-IDF)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# 1. Separate clustered articles (exclude noise label -1)
clustered_df = df[df['subcategory_id'] != -1].copy()

# 2. Group all articles in each cluster into one big document
cluster_documents = clustered_df.groupby('subcategory_id')['cleaned_text'].apply(
    lambda texts: ' '.join(texts)
).reset_index()

# 3. c-TF-IDF — Compare word frequencies ACROSS clusters
# This finds words that are unique to each cluster (not just common everywhere)
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2),    # Include bigrams like "world cup", "premier league"
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(cluster_documents['cleaned_text'])
feature_names = vectorizer.get_feature_names_out()

# 4. Extract top keywords per cluster
def get_top_keywords(tfidf_matrix, feature_names, n=10):
    top_keywords = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray()[0]
        top_indices = row.argsort()[-n:][::-1]
        keywords = [feature_names[idx] for idx in top_indices]
        top_keywords.append(keywords)
    return top_keywords

top_keywords = get_top_keywords(tfidf_matrix, feature_names, n=10)

# 5. Display keywords per cluster
print(" Top keywords per subcategory:\n")
for i, row in cluster_documents.iterrows():
    cluster_id = row['subcategory_id']
    keywords = top_keywords[i]
    print(f"Cluster {cluster_id:2d} | {', '.join(keywords)}")

# 6. Add top 3 keywords as semantic label to DataFrame
cluster_documents['semantic_label'] = [
    ' | '.join(kws[:3]) for kws in top_keywords
]

# 7. Map labels back to main DataFrame
label_map = dict(zip(cluster_documents['subcategory_id'], cluster_documents['semantic_label']))
df['semantic_label'] = df['subcategory_id'].map(label_map)
df['semantic_label'] = df['semantic_label'].fillna('Noise')

print(f"\n Semantic labels assigned!")
print(f"\n Cluster ID → Semantic Label:")
for cluster_id, label in sorted(label_map.items()):
    print(f"  Cluster {cluster_id:2d} → {label}")

 Top keywords per subcategory:

Cluster  0 | indoor, holmes, olympic, champion, world, 60m, marathon, year, 000m, race
Cluster  1 | roddick, seed, open, said, nadal, australian open, tennis, australian, hewitt, henman
Cluster  2 | england, kaplan, robinson, referee, ireland, lewsey, cueto, nations, game, campese
Cluster  3 | injury, chelsea, robben, barcelona, gallas, said, weeks, mourinho, play, bridge
Cluster  4 | said, arsenal, game, aragones, referee, chelsea, racist, united, incident, henry
Cluster  5 | kenteris, thanou, iaaf, greek, athens, tests, charges, athletics, tribunal, drugs
Cluster  6 | conte, balco, doping, drugs, enhancing, enhancing drugs, performance enhancing, jones, anti doping, marion
Cluster  7 | club, liverpool, parry, gerrard, said, gbp, new, steven, contract, deal
Cluster  8 | england, rugby, wales, nations, new zealand, zealand, said, robinson, lions, players
Cluster  9 | wales, france, england, laporte, henson, nations, stade, half, williams, jones
Cluster 1

## Phase 5b: Article-Level Keyword Extraction (KeyBERT)
In this phase we extract article-level keywords using KeyBERT. Unlike c-TF-IDF which operates at the cluster level, KeyBERT extracts keywords that best represent each individual article — providing a richer and more explainable output.

In [6]:
# ============================================================
# PHASE 5B: KEYBERT + CLUSTER VERIFICATION
# ============================================================
from keybert import KeyBERT
import warnings
warnings.filterwarnings('ignore')

# 1. KeyBERT keyword extraction
print(" Loading KeyBERT model...")
kw_model = KeyBERT()
print(" KeyBERT model loaded!\n")

print(" Extracting keywords for each article...")
def extract_keywords(text, n=5):
    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=n
        )
        return ', '.join([kw[0] for kw in keywords])
    except:
        return ''

df['article_keywords'] = df['cleaned_text'].apply(extract_keywords)
print(f" Keywords extracted for all {len(df)} articles!\n")

# 2. Cluster verification — read before naming
print("=" * 70)
print("      CLUSTER VERIFICATION — READ BEFORE NAMING")
print("=" * 70)

for cluster_id in sorted(df[df['subcategory_id'] != -1]['subcategory_id'].unique()):
    cluster_articles = df[df['subcategory_id'] == cluster_id]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id} — {len(cluster_articles)} articles")
    print(f"{'='*70}")
    
    # Sample articles — focus on cleaned_text
    samples = cluster_articles.sample(min(3, len(cluster_articles)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n Title {i}: {row['title']}")
        print(f" Keywords: {row['article_keywords']}")
        snippet = ' '.join(str(row['cleaned_text']).split()[:80])
        print(f" Text: {snippet}...")
    
    print(f"\n  WHAT LABEL SHOULD THIS CLUSTER GET?")
    print(f"{'='*70}")

 Loading KeyBERT model...
 KeyBERT model loaded!

 Extracting keywords for each article...
 Keywords extracted for all 511 articles!

      CLUSTER VERIFICATION — READ BEFORE NAMING

CLUSTER 0 — 66 articles

 Title 1: Pavey focuses on indoor success
 Keywords: jo pavey, pavey miss, training pavey, pavey, pavey came
 Text: Jo Pavey will miss January's View From Great Edinburgh International Cross Country to focus on preparing for the European Indoor Championships in March. The 31-year-old was third behind Hayley Yelling and Justyna Bak in last week's European Cross Country Championships but she prefers to race on the track. "It was great winning bronze but I'm wary of injuries and must concentrate on the indoor season," she said. "Because of previous injuries I don't even run up hills in training." Pavey,...

 Title 2: Disappointed Scott in solid start
 Keywords: slower iaaf, won felipe, athlete race, ran faster, iaaf
 Text: Allan Scott is confident of winning a medal at next week's Eur

## Phase 6: Manual Label Assignment & Final Output
In this phase we assign professional subcategory names to each cluster using AI-generated labels based on the top c-TF-IDF keywords. The final output DataFrame is built and saved to CSV with all required columns.

In [ ]:
# ============================================================
## Phase 6: Manual Label Assignment & Final Output
# ============================================================

# 1. AI-generated subcategory names for 14 clusters
ai_label_map = {
    0:  "Athletics",
    1:  "Tennis",
    2:  "Rugby Officiating",
    3:  "Player Injuries",
    4:  "Premier League",
    5:  "Doping Governance",
    6:  "Doping Scandals",
    7:  "Transfer News",
    8:  "International Rugby",
    9:  "Six Nations",
    10: "Six Nations Results",
    11: "Domestic Cups",
    12: "Premiership Analysis",
    13: "Champions League",
    -1: "Noise"
}

# 2. Apply confidence scores
df['confidence_score'] = confidence_scores
df.loc[df['subcategory_id'] == -1, 'confidence_score'] = 0.0

# 3. Apply AI labels
df['semantic_label'] = df['subcategory_id'].map(ai_label_map)

# 4. Build final output DataFrame cleanly
final_df = df[[
    'file_name',
    'article_id',
    'title',
    'cleaned_text',
    'subcategory_id',
    'semantic_label',
    'confidence_score',
    'article_keywords'
]].copy().reset_index(drop=True)

# 5. Save to CSV
final_df.to_csv('sport_subcategories_output.csv', index=False)

# 6. Summary
print("=" * 55)
print("      SPORT CLASSIFIER — FINAL RESULTS")
print("=" * 55)
print(f" Total Articles:        {len(final_df)}")
print(f" Subcategories Found:   {final_df[final_df['subcategory_id'] != -1]['subcategory_id'].nunique()}")
print(f" Clustered Articles:    {len(final_df[final_df['subcategory_id'] != -1])}")
print(f"  Noise Articles:        {len(final_df[final_df['subcategory_id'] == -1])}")
print(f" Avg Confidence:        {final_df[final_df['subcategory_id'] != -1]['confidence_score'].mean():.2f}")
print("=" * 55)
print(f"\n Label Distribution:")
print(final_df['semantic_label'].value_counts())
print(f"\n Saved to sport_subcategories_output.csv")

      SPORT CLASSIFIER — FINAL RESULTS
 Total Articles:        511
 Subcategories Found:   14
 Clustered Articles:    459
  Noise Articles:        52
 Avg Confidence:        0.94

 Label Distribution:
semantic_label
Tennis                  88
International Rugby     68
Athletics               66
Transfer News           52
Noise                   52
Six Nations Results     33
Premier League          29
Champions League        24
Six Nations             22
Domestic Cups           19
Doping Governance       17
Player Injuries         12
Doping Scandals         10
Rugby Officiating       10
Premiership Analysis     9
Name: count, dtype: int64

 Saved to sport_subcategories_output.csv


## Phase 8: Manual Noise Correction
Noise articles reviewed in Phase 7 are manually reassigned here based on their content, keywords, and title. Each article is mapped to the most appropriate subcategory and the updated labels are saved back to the output CSV.

In [9]:
# ============================================================
# PHASE 8: MANUAL NOISE CORRECTION
# ============================================================
import pandas as pd

# Load the CSV
df_final = pd.read_csv('sport_subcategories_output.csv')

# All reassignments — original 14 + noise corrections
noise_assignments = {

    # Football (misclassified as Rugby Union)
    99:  "Football",
    113: "Football",
    115: "Football",
    119: "Football",
    128: "Football",
    129: "Football",
    132: "Football",
    133: "Football",
    134: "Football",
    208: "Football",
    212: "Football",
    217: "Football",

    # Rugby League (misclassified as Rugby Union)
    397: "Rugby League",
    399: "Rugby League",

    # Doping (from noise)
    26:  "Doping",
    35:  "Doping",
    36:  "Doping",
    39:  "Doping",
    44:  "Doping",
    45:  "Doping",
    46:  "Doping",
    49:  "Doping",
    88:  "Doping",
    91:  "Doping",
    199: "Doping",
    201: "Doping",
    203: "Doping",
    448: "Doping",
    461: "Doping",
    485: "Doping",
    502: "Doping",

    # Football Match Reports (from noise)
    122: "Football Match Reports",
    179: "Football Match Reports",
    180: "Football Match Reports",
    184: "Football Match Reports",
    192: "Football Match Reports",
    205: "Football Match Reports",

    # Football (from noise)
    123: "Football",
    135: "Football",
    144: "Football",
    160: "Football",
    161: "Football",
    164: "Football",
    190: "Football",
    194: "Football",
    202: "Football",
    207: "Football",
    215: "Football",
    219: "Football",
    221: "Football",
    227: "Football",
    242: "Football",
    245: "Football",
    247: "Football",
    260: "Football",
    263: "Football",
    267: "Football",
    275: "Football",
    277: "Football",

    # Football Transfers (from noise)
    165: "Football Transfers",

    # Champions League (from noise)
    237: "Champions League",

    # Rugby Union (from noise)
    293: "Rugby Union",
    294: "Rugby Union",
    295: "Rugby Union",
    310: "Rugby Union",
    317: "Rugby Union",
    326: "Rugby Union",
    353: "Rugby Union",
    363: "Rugby Union",
    376: "Rugby Union",
    381: "Rugby Union",
    384: "Rugby Union",
    394: "Rugby Union",
    400: "Rugby Union",
    417: "Rugby Union",

    # Rugby League (from noise)
    398: "Rugby League",
}

# Apply assignments
for article_id, label in noise_assignments.items():
    mask = df_final['article_id'] == article_id
    df_final.loc[mask, 'semantic_label'] = label

# Save updated CSV
df_final.to_csv('sport_subcategories_output.csv', index=False)

# Summary
remaining_noise = len(df_final[df_final['semantic_label'] == 'Noise'])
print("=" * 55)
print("      PHASE 8 — MANUAL NOISE CORRECTION")
print("=" * 55)
print(f" Manually assigned:     {len(noise_assignments)}")
print(f" Remaining noise:       {remaining_noise}")
print(f" Total classified:      {len(df_final[df_final['semantic_label'] != 'Noise'])}")
print(f"\n Final Label Distribution:")
print(df_final['semantic_label'].value_counts())
print("=" * 55)

      PHASE 8 — MANUAL NOISE CORRECTION
 Manually assigned:     76
 Remaining noise:       0
 Total classified:      511

 Final Label Distribution:
semantic_label
Tennis                    88
Athletics                 66
International Rugby       63
Transfer News             52
Football                  34
Premier League            29
Champions League          25
Six Nations Results       24
Six Nations               22
Domestic Cups             19
Doping                    17
Doping Governance         17
Rugby Union               14
Player Injuries           12
Rugby Officiating         10
Premiership Analysis       9
Football Match Reports     6
Rugby League               3
Football Transfers         1
Name: count, dtype: int64


## Phase 8A: Merge Small Subcategories

Three subcategories had insufficient article counts to stand alone and were merged into semantically appropriate parent categories.

| From                | Into          |
|---------------------|---------------|
| Football Transfers  | Football      |
| Football Match Reports | Domestic Cups |
| Rugby League        | Rugby Union   |

In [10]:
merge_map = {
    "Football Transfers": "Football",
    "Football Match Reports": "Domestic Cups",
    "Rugby League": "Rugby Union"
}

for old_label, new_label in merge_map.items():
    df_final.loc[df_final['semantic_label'] == old_label, 'semantic_label'] = new_label

df_final.to_csv('sport_subcategories_output.csv', index=False)

print(df_final['semantic_label'].value_counts())

semantic_label
Tennis                  88
Athletics               66
International Rugby     63
Transfer News           52
Football                35
Premier League          29
Domestic Cups           25
Champions League        25
Six Nations Results     24
Six Nations             22
Doping Governance       17
Doping                  17
Rugby Union             17
Player Injuries         12
Rugby Officiating       10
Premiership Analysis     9
Name: count, dtype: int64


## Phase 8c: Merge Doping Governance into Doping

In this step we consolidate two closely related subcategories into one. **Doping Governance** articles cover institutional proceedings, tribunal hearings and WADA/IAAF decisions — all of which are part of the broader doping topic. Merging avoids thin category splits and produces a more statistically meaningful subcategory.

In [11]:
df_final.loc[df_final['semantic_label'] == 'Doping Governance', 'semantic_label'] = 'Doping'

df_final.to_csv('sport_subcategories_output.csv', index=False)

print(df_final['semantic_label'].value_counts())

semantic_label
Tennis                  88
Athletics               66
International Rugby     63
Transfer News           52
Football                35
Doping                  34
Premier League          29
Domestic Cups           25
Champions League        25
Six Nations Results     24
Six Nations             22
Rugby Union             17
Player Injuries         12
Rugby Officiating       10
Premiership Analysis     9
Name: count, dtype: int64


## Phase 9: Named Entity Recognition with April Events Extraction
In this phase we address the project's **Desired** requirement from the project brief:

> *"Identify documents and extract the named entities for media personalities, clearly identifying their jobs (e.g. Politicians, TV/Film Personalities, Musicians)"*

> *"Extract summaries of anything that took place or is/was scheduled to take place in April."*

We use spaCy to extract named entities from each article's combined `title` and `cleaned_text` for the **sport category** (511 articles). First we verify spaCy is installed and the English language model is available.

## Phase 9A: Named Entity Recognition — Persons, Organisations, Locations (Sport Category)
In this phase we extract named entities from each article's combined `title` and `cleaned_text` using spaCy's `en_core_web_lg` model across all 511 sport articles. Three entity types are extracted:

- **Named Persons** — validated through a false positive filter that removes sports terms, movie titles, month names, acronyms, and single-word misclassifications, ensuring only real two-word-minimum person names are retained
- **Named Organisations** — clubs, sporting bodies, federations, and other organisations mentioned
- **Named Locations** — countries, cities, stadiums, and regions identified as GPE or LOC entities

Each valid person is also classified by role using a context window of 80 characters surrounding the entity, matching against keyword lists for roles including Politician, Business Executive, Expert/Analyst, TV/Film Personality, Musician, Sports Personality, Journalist, and Other. Results: **509** articles with named persons, **492** with organisations, **492** with locations.

In [16]:
# ============================================================
# PHASE 9A: NER — PERSONS, ORGS, LOCATIONS (SPORT CATEGORY)
# ============================================================
import spacy
import pandas as pd

nlp = spacy.load('en_core_web_lg')
df = pd.read_csv('sport_subcategories_output.csv')
print(f" Loaded {len(df)} sport articles")

# ============================================================
# FALSE POSITIVE FILTER
# ============================================================
FALSE_POSITIVES = {
    # Sports terms misclassified as persons
    'champions', 'premier', 'olympic', 'paralympic',
    'world cup', 'grand slam', 'super bowl',
    # Movie/TV titles
    'alexander', 'catwoman', 'rings', 'halo', 'titanic',
    'batman', 'superman', 'spiderman', 'matrix', 'avatar',
    # Days/Months
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday',
    'saturday', 'sunday', 'january', 'february', 'march',
    'april', 'may', 'june', 'july', 'august', 'september',
    'october', 'november', 'december',
    # Common misclassified words
    'lord', 'sir', 'mr', 'mrs', 'ms', 'dr', 'prof',
    'aol', 'sec', 'gm', 'ibm', 'bbc', 'itv',
}

# Exact semantic_label values from sport dataset
SPORT_LABELS = {
    'tennis', 'athletics', 'international rugby', 'transfer news',
    'football', 'doping', 'premier league', 'domestic cups',
    'champions league', 'six nations results', 'six nations',
    'rugby union', 'player injuries', 'rugby officiating',
    'premiership analysis'
}

def is_valid_person(name):
    if len(name) < 2:
        return False
    if name.lower() in FALSE_POSITIVES:
        return False
    if name.isupper():
        return False
    if not any(c.isalpha() for c in name):
        return False
    tokens = name.strip().split()
    if len(tokens) < 2:
        return False
    # filter possessives
    if name.endswith("'s") or name.endswith("s'"):
        return False
    # filter honorific-only entries
    honorifics = {'mr', 'mrs', 'ms', 'dr', 'prof', 'dame', 'sir', 'lord'}
    if tokens[0].lower().rstrip('.') in honorifics and len(tokens) == 2:
        return False
    # filter single initials like "J McIlroy", "K Holmes"
    if len(tokens[0]) == 1 and tokens[0].isupper():
        return False
    # filter scorecard entries like "Rooney 87", "Terry 17"
    if tokens[-1].isdigit():
        return False
    # filter comma-separated pairs like "Hayes, O'Kelly"
    if ',' in name:
        return False
    # filter entries with dots like "S. Young", "AJ Venter"
    if len(tokens[0]) <= 2 and '.' in tokens[0]:
        return False
    return True

# ============================================================
# KNOWN PERSON ROLE LOOKUP
# ============================================================
KNOWN_ROLES = {
    # Football managers/coaches
    'Bobby Robson':       'Sports Personality',
    'Phil Thompson':      'Sports Personality',
    'Gerard Houllier':    'Sports Personality',
    'Alastair Campbell':  'Journalist',
    # Rugby
    'Martin Johnson':     'Sports Personality',
    'Lawrence Dallaglio': 'Sports Personality',
    'Jason Robinson':     'Sports Personality',
    'Jonny Wilkinson':    'Sports Personality',
    'Austin Healey':      'Sports Personality',
    'Phil Vickery':       'Sports Personality',
    'Clive Woodward':     'Sports Personality',
    'Lawrence Dallaglio': 'Sports Personality',
    # Tennis
    'Lindsay Davenport':  'Sports Personality',
    'Andy Williams':      'Sports Personality',
}

# ============================================================
# PERSON ROLE CLASSIFIER
# ============================================================
def classify_role(person, context, semantic_label=''):
    # Check known persons first
    if person in KNOWN_ROLES:
        return KNOWN_ROLES[person]

    context = context.lower()
    semantic_label = semantic_label.lower().strip() if semantic_label else ''

    if any(w in context for w in [
        'president', 'prime minister', 'minister', 'senator', 'mp ',
        'chancellor', 'governor', 'secretary of state', 'politician', 'parliament'
    ]):
        return 'Politician'
    elif any(w in context for w in [
        'lawyer', 'attorney', 'legal counsel', 'defence counsel',
        'defendant', 'prosecution', 'lawsuit', 'sued', 'trial',
        'verdict', 'indicted', 'charged', 'acquitted', 'solicitor', 'barrister'
    ]):
        return 'Lawyer'
    elif any(w in context for w in [
        'actor', 'actress', 'director', 'film', 'movie', 'tv', 'television',
        'presenter', 'host', 'comedian', 'starred', 'cast', 'role', 'screen'
    ]):
        return 'TV/Film Personality'
    elif any(w in context for w in [
        'singer', 'musician', 'rapper', 'band', 'music', 'album', 'song',
        'artist', 'record', 'chart', 'tour', 'concert', 'hip hop', 'rap'
    ]):
        return 'Musician'
    elif any(w in context for w in [
        'coach', 'manager', 'player', 'footballer', 'athlete', 'champion',
        'scored', 'wicket', 'serve', 'sprint', 'tackle', 'referee',
        'tennis', 'cricket', 'rugby', 'golf', 'swimming', 'cycling',
        'boxing', 'athletics', 'race', 'tournament', 'league', 'match',
        'fixture', 'transfer', 'squad', 'team', 'club', 'olympic',
        'medal', 'gold', 'silver', 'bronze', 'world record', 'championship',
        'hurdler', 'jumper', 'sprinter', 'runner', 'striker', 'defender',
        'goalkeeper', 'batsman', 'bowler', 'jockey', 'trainer', 'doping',
        'anti-doping', 'ban', 'suspension', 'test positive', 'drug test',
        'set', 'break', 'ace', 'deuce', 'volley', 'forehand', 'backhand',
        'grand slam', 'wimbledon', 'australian open', 'davis cup',
        'six nations', 'premiership', 'line-out', 'scrum', 'try', 'prop',
        'hooker', 'flanker', 'fly-half', 'lock', 'wing', 'fullback',
        'cap', 'capped', 'international', 'lions', 'test match'
    ]):
        return 'Sports Personality'
    elif any(w in context for w in [
        'ceo', 'chief executive', 'chairman', 'founder', 'executive',
        'managing director', 'mogul', 'tycoon', 'entrepreneur',
        'investor', 'billionaire', 'company', 'corporation'
    ]):
        return 'Business Executive'
    elif any(w in context for w in [
        'analyst', 'economist', 'researcher', 'professor', 'scientist',
        'expert', 'academic', 'lecturer', 'scholar'
    ]):
        return 'Expert/Analyst'
    elif any(w in context for w in [
        'journalist', 'reporter', 'editor', 'correspondent', 'critic',
        'columnist', 'broadcaster', 'commentator', 'pundit', 'anchor'
    ]):
        return 'Journalist'
    # Subcategory fallback — all sport labels default to Sports Personality
    elif semantic_label in SPORT_LABELS:
        return 'Sports Personality'
    else:
        return 'Other'

# ============================================================
# NER EXTRACTION FUNCTION
# ============================================================
def extract_persons_orgs_locations(title, cleaned_text, semantic_label=''):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    persons = []
    person_roles = []
    orgs = []
    locations = []

    seen_persons = set()
    seen_orgs = set()
    seen_locations = set()

    for ent in doc.ents:

        if ent.label_ == 'PERSON':
            if ent.text not in seen_persons and is_valid_person(ent.text):
                seen_persons.add(ent.text)
                start = max(0, ent.start_char - 300)
                end = min(len(combined), ent.end_char + 300)
                context = combined[start:end]
                role = classify_role(ent.text, context, semantic_label)
                persons.append(ent.text)
                person_roles.append(f"{ent.text}: {role}")

        elif ent.label_ == 'ORG':
            if ent.text not in seen_orgs:
                seen_orgs.add(ent.text)
                orgs.append(ent.text)

        elif ent.label_ in ['GPE', 'LOC']:
            if ent.text not in seen_locations:
                seen_locations.add(ent.text)
                locations.append(ent.text)

    return {
        'named_persons':       ' | '.join(persons),
        'named_organisations': ' | '.join(orgs),
        'named_locations':     ' | '.join(locations),
        'person_roles':        ' | '.join(person_roles),
    }

# ============================================================
# RUN NER ON ALL ARTICLES
# ============================================================
print(" Running NER (Persons, Orgs, Locations)...")

named_persons       = []
named_organisations = []
named_locations     = []
person_roles        = []

for i, row in df.iterrows():
    result = extract_persons_orgs_locations(
        str(row['title']),
        str(row['cleaned_text']),
        str(row.get('semantic_label', ''))
    )
    named_persons.append(result['named_persons'])
    named_organisations.append(result['named_organisations'])
    named_locations.append(result['named_locations'])
    person_roles.append(result['person_roles'])

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles

print(f"\n Person/Org/Location NER complete!")
print(f" Articles with Persons:   {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:      {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations: {sum(1 for x in named_locations if x)}")

# ============================================================
# QUICK AUDIT — check directly on dataframe
# ============================================================
other_count = df['person_roles'].str.contains('Other', na=False).sum()
print(f" Articles still containing 'Other': {other_count}")

other_rows = df[df['person_roles'].str.contains("Other", na=False)].copy()

def extract_others(person_roles_str):
    entries = person_roles_str.split(" | ")
    return [e.strip() for e in entries if e.strip().endswith(": Other")]

other_rows['other_persons'] = other_rows['person_roles'].apply(extract_others)

for _, row in other_rows.iterrows():
    print(f"Article {row['article_id']} | {row['title'][:60]}")
    for person in row['other_persons']:
        print(f"   --> {person}")
    print()

 Loaded 511 sport articles
 Running NER (Persons, Orgs, Locations)...
   Processed 50/511 articles...
   Processed 100/511 articles...
   Processed 150/511 articles...
   Processed 200/511 articles...
   Processed 250/511 articles...
   Processed 300/511 articles...
   Processed 350/511 articles...
   Processed 400/511 articles...
   Processed 450/511 articles...
   Processed 500/511 articles...

 Person/Org/Location NER complete!
 Articles with Persons:   509
 Articles with Orgs:      492
 Articles with Locations: 492
 Articles still containing 'Other': 0


## Phase 9B: April Events Extraction (Sport Category)
In this phase we extract April-related events from each article using spaCy's DATE entity recognition. For every article where a DATE entity containing "April" is detected in the combined `title` and `cleaned_text`, a 300-character contextual window surrounding the mention is captured as the event summary. This satisfies the project's Desired requirement of identifying events that took place or were scheduled to take place in April. Results: **19 out of 511** sport articles contained April events.

In [17]:
# ============================================================
# PHASE 9B: NER — APRIL EVENTS (SPORT CATEGORY)
# ============================================================

def extract_april_events(title, cleaned_text):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    april_events = []

    for ent in doc.ents:
        if ent.label_ == 'DATE':
            if 'april' in ent.text.lower():
                start = max(0, ent.start_char - 100)
                end = min(len(combined), ent.end_char + 200)
                april_events.append(combined[start:end].strip())

    return ' | '.join(april_events) if april_events else ''

print(" Extracting April events...")

april_events = []
for i, row in df.iterrows():
    result = extract_april_events(str(row['title']), str(row['cleaned_text']))
    april_events.append(result)

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['april_events'] = april_events

print(f"\n April Events extraction complete!")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")

 Extracting April events...
   Processed 50/511 articles...
   Processed 100/511 articles...
   Processed 150/511 articles...
   Processed 200/511 articles...
   Processed 250/511 articles...
   Processed 300/511 articles...
   Processed 350/511 articles...
   Processed 400/511 articles...
   Processed 450/511 articles...
   Processed 500/511 articles...

 April Events extraction complete!
  Articles with April Events: 19


## Phase 9C: Save Complete NER Output (Sport Category)
In this phase we consolidate all extracted NER columns — `named_persons`, `named_organisations`, `named_locations`, `person_roles`, and `april_events` — into the final dataframe and save to `sport_ner_output.csv`. A sample verification of the first article is printed to confirm all columns are correctly populated before proceeding to the next category.

In [18]:
# ============================================================
# PHASE 9C: SAVE COMPLETE NER OUTPUT (SPORT CATEGORY)
# ============================================================

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles
df['april_events']        = april_events

df.to_csv('sport_ner_output.csv', index=False)
print(" Saved to sport_ner_output.csv")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample row 1:")
print(f" Persons:  {df['named_persons'].iloc[0]}")
print(f" Orgs:     {df['named_organisations'].iloc[0]}")
print(f" Location: {df['named_locations'].iloc[0]}")
print(f" Roles:    {df['person_roles'].iloc[0]}")
print(f"  April:    {df['april_events'].iloc[0]}")

print("\n" + "=" * 55)
print("        SPORT NER — FINAL SUMMARY")
print("=" * 55)
print(f" Total Articles:             {len(df)}")
print(f" Articles with Persons:      {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:         {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations:    {sum(1 for x in named_locations if x)}")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")
print("=" * 55)

 Saved to sport_ner_output.csv
 Columns: ['file_name', 'article_id', 'title', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords', 'cleaned_text', 'named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events']

 Sample row 1:
 Persons:  Sarah Claxton | Irina Shevchenko
 Orgs:     European Indoor Championships | AAAs
 Location: Madrid | Scotland | Colchester | London
 Roles:    Sarah Claxton: Musician | Irina Shevchenko: Politician
  April:    

        SPORT NER — FINAL SUMMARY
 Total Articles:             511
 Articles with Persons:      509
 Articles with Orgs:         492
 Articles with Locations:    492
  Articles with April Events: 19
